## Imports

In [38]:
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification
from peft import LoraConfig, get_peft_model, TaskType
import torch
import pandas as pd

## Load dataset

In [39]:
df = pd.read_csv("quality_data.csv")

## Label creation

In [40]:
df["label"] = df["category"]
labels = df["label"].unique().tolist()
label2id = {label: i for i, label in enumerate(labels)}
id2label = {i: label for label, i in label2id.items()}

df["label_id"] = df["label"].map(label2id)

In [41]:
df["input_text"] = df["narration"] + " " + df["mode"] + " " + df["type"]

## Tokenizer

In [42]:
tokenizer = DistilBertTokenizer.from_pretrained("distilbert-base-uncased")

In [43]:
from datasets import Dataset

# Step 1: Create dataset FIRST
dataset = Dataset.from_pandas(df[["input_text", "label_id"]])

# Step 2: Tokenization
def tokenize_function(example):
    return tokenizer(
        example["input_text"],
        padding="max_length",
        truncation=True,
        max_length=64
    )

dataset = dataset.map(tokenize_function)

# Step 3: Rename label column
dataset = dataset.rename_column("label_id", "labels")

# Step 4: Torch format
dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

Map:   0%|          | 0/198 [00:00<?, ? examples/s]

## MODEL + LORA

In [44]:
base_model = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=len(labels)
)

lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=8,
    lora_alpha=16,
    lora_dropout=0.1,
    bias="none",

    # 🔥 THIS IS THE FIX
    target_modules=["q_lin", "v_lin"]
)

model = get_peft_model(base_model, lora_config)

# 🔥 important (for visibility)
model.print_trainable_parameters()

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


trainable params: 740,355 || all params: 67,696,134 || trainable%: 1.0936


In [45]:
from datasets import Dataset

# 1. Create dataset
dataset = Dataset.from_pandas(df[["input_text", "label_id"]])

# 2. Tokenize
dataset = dataset.map(tokenize_function)

# 3. Rename
dataset = dataset.rename_column("label_id", "labels")

# 4. Torch format
dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

# 5. Split
dataset = dataset.train_test_split(test_size=0.2)

train_dataset = dataset["train"]
test_dataset = dataset["test"]

Map:   0%|          | 0/198 [00:00<?, ? examples/s]

In [46]:
from torch.utils.data import DataLoader

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=16)

In [47]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model.to(device)

PeftModelForSequenceClassification(
  (base_model): LoraModel(
    (model): DistilBertForSequenceClassification(
      (distilbert): DistilBertModel(
        (embeddings): Embeddings(
          (word_embeddings): Embedding(30522, 768, padding_idx=0)
          (position_embeddings): Embedding(512, 768)
          (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (dropout): Dropout(p=0.1, inplace=False)
        )
        (transformer): Transformer(
          (layer): ModuleList(
            (0-5): 6 x TransformerBlock(
              (attention): MultiHeadSelfAttention(
                (dropout): Dropout(p=0.1, inplace=False)
                (q_lin): lora.Linear(
                  (base_layer): Linear(in_features=768, out_features=768, bias=True)
                  (lora_dropout): ModuleDict(
                    (default): Dropout(p=0.1, inplace=False)
                  )
                  (lora_A): ModuleDict(
                    (default): Linear(in_features=768

In [48]:
model.train()

optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5)

epochs = 8

for epoch in range(epochs):
    print(f"\nEpoch {epoch+1}")

    total_loss = 0

    for batch in train_loader:
        batch = {k: v.to(device) for k, v in batch.items()}

        outputs = model(**batch)
        loss = outputs.loss

        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        total_loss += loss.item()

    print(f"Loss: {total_loss / len(train_loader)}")


Epoch 1
Loss: 1.0324890077114106

Epoch 2
Loss: 0.9109265267848968

Epoch 3
Loss: 0.863743805885315

Epoch 4
Loss: 0.8291514039039611

Epoch 5
Loss: 0.7764559686183929

Epoch 6
Loss: 0.7336285322904587

Epoch 7
Loss: 0.6679727613925934

Epoch 8
Loss: 0.5942419230937958


## Evaluation

In [49]:
from sklearn.metrics import classification_report
import numpy as np

model.eval()

all_preds = []
all_labels = []

with torch.no_grad():
    for batch in test_loader:
        batch = {k: v.to(device) for k, v in batch.items()}

        outputs = model(**batch)
        logits = outputs.logits

        preds = torch.argmax(logits, dim=1)

        all_preds.extend(preds.cpu().tolist())
        all_labels.extend(batch["labels"].cpu().tolist())

In [50]:
pred_labels = [id2label[p] for p in all_preds]
true_labels = [id2label[l] for l in all_labels]

In [51]:
print(classification_report(true_labels, pred_labels))

              precision    recall  f1-score   support

     expense       0.69      1.00      0.81        24
      income       0.50      0.50      0.50         2
  investment       1.00      0.21      0.35        14

    accuracy                           0.70        40
   macro avg       0.73      0.57      0.56        40
weighted avg       0.79      0.70      0.64        40



## Prediction loop

In [52]:
def predict(text):
    model.eval()

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=64
    ).to(device)

    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits

    probs = torch.softmax(logits, dim=1)[0]
    pred_idx = torch.argmax(probs).item()

    label = id2label[pred_idx]
    category = label
    subcategory = None

    confidence = probs[pred_idx].item()

    return category, subcategory, confidence

In [53]:
text = df["input_text"].iloc[0]

pred_category, pred_subcategory, confidence = predict(text)

In [54]:
from sklearn.metrics import fbeta_score

f2 = fbeta_score(true_labels, pred_labels, average='weighted', beta=2)
print("F2 Score:", round(f2, 3))

F2 Score: 0.664


In [55]:
for i in range(10):
    print("TEXT:", df["input_text"].iloc[i])
    print("TRUE:", true_labels[i])
    print("PRED:", pred_labels[i])
    print("-----")

TEXT: emi auto debit sbi acc AUTO_DEBIT Debit
TRUE: expense
PRED: expense
-----
TEXT: salary credited via hdfc NEFT Credit
TRUE: investment
PRED: expense
-----
TEXT: upi txn to zomato UPI Debit
TRUE: expense
PRED: expense
-----
TEXT: paid electricity bill online NETBANKING Debit
TRUE: investment
PRED: income
-----
TEXT: mutual fund sip deducted AUTO_DEBIT Debit
TRUE: expense
PRED: expense
-----
TEXT: got interest from fd NEFT Credit
TRUE: investment
PRED: investment
-----
TEXT: amazon pay txn UPI Debit
TRUE: income
PRED: expense
-----
TEXT: credited bonus amount NEFT Credit
TRUE: investment
PRED: expense
-----
TEXT: fuel payment hp petrol UPI Debit
TRUE: investment
PRED: expense
-----
TEXT: insurance premium paid AUTO_DEBIT Debit
TRUE: expense
PRED: expense
-----
